In [23]:
!pip install -qU  duckduckgo-search langchain-community


In [24]:
!pip install langchain-groq


In [25]:
!pip install -U ddgs

In [26]:
from google.colab import userdata
groq_api_key=userdata.get("GROQ_API_KEY")
from langchain_groq import ChatGroq
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model="qwen/qwen3.6-27b",
    temperature=0.7,
)

# Tools

## DUCKDUCKGOSEARCH TOOL

In [27]:
from  langchain_community.tools import DuckDuckGoSearchRun
def search_duckduckgo(query:str) ->str:
  """This tool search the latest news on DuckDuckGo for the given query  and returns the results"""
  duck_search = DuckDuckGoSearchRun()
  return duck_search.invoke(query)
#duck_search = DuckDuckGoSearchRun()
#duck_search.invoke("Tranding News About LangChain and LangGraph ")

### so this answer not generated by llm there is no llm involed in the above code ,this response directly get from duckduckgosearchRun


###So if my llm  dosen't have any latest information ,latest knowledge about any topic  I can use this tool to get the latest information then I can provide  this to LLM,then LLM use this information ,that provide the run time information during llm calls

# ARXIV Query Tool

In [28]:
!pip install pymupdf

In [ ]:
!pip install -q -U langchain_community

In [ ]:
!pip show arxiv | grep Version

In [ ]:
!pip uninstall -y arxiv
!pip install -q "arxiv==1.4.8"
!pip install -q -U arxiv

In [ ]:
import arxiv
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun
from langchain.tools import tool
@tool
def arxiv_tool(query:str)->str:
  """Use this tool to search for papers on Arxiv."""
  arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
  return arxiv_query.invoke(query)
"""
# Restore Search.results() using the new Client-based API (which correctly uses https)
_client = arxiv.Client()

def _patched_results(self, offset=0):
    return _client.results(self, offset=offset)

arxiv.Search.results = _patched_results

# Confirm it's using https now
print(arxiv.Client)  # sanity check import works

arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
result = arxiv_query.invoke("Transformer models in NLP")
print(result)"""

# Wikipedia Search Tool

In [ ]:
!pip install wikipedia

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.tools import tool
@tool
def wiki_tool(query:str):
  """Use this tool to get information from Wikipedia."""
  wiki_query=WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
  return wiki_query.invoke(query)
#wiki_query=WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
#wiki_query.invoke("what is Langraph")

# Custom Tool

In [ ]:
from langchain.tools import tool
@tool
def person_info_tool(name:str):
   """Use this custom tool to get personal information about Alice,Bob,Charlie."""
   info={
       "Alice":"Alice is a software engineer with 5 years of experience in AI.",
       "Bob":"Bob is a data scientist who loves working with large datasets.",
       "Charlie":"Charlie is a product manager with a background in tech startups."
   }

   return info.get(name,"No information avaiable for this person.")

In [ ]:
person_info_tool.invoke("Alice")

# Tool Binding with LLM

In [ ]:
tools=[search_duckduckgo,arxiv_tool,person_info_tool]
llm_with_tools=llm.bind_tools(tools)

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

# Ensure groq_api_key is available from the kernel state
# groq_api_key = userdata.get("GROQ_API_KEY")

# Re-initialize llm with a max_tokens limit to avoid rate limit error
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model="qwen/qwen3.6-27b",
    temperature=0.7,
    max_tokens=500 # Limit output tokens to prevent RateLimitError
)

# Ensure tools are available from the kernel state
# tools=[search_duckduckgo,arxiv_tool,person_info_tool]

# Rebind tools to the newly configured llm instance
llm_with_tools=llm.bind_tools(tools)

response=llm_with_tools.invoke("what is the latest news on AI")
response.tool_calls

Here LLm not giving the answer bcz lm not access with LATEST NEWS ...but it is ot just a llm it is llm with tools

Now LLM is trying to say Okay I cannot generate this thing but  I can Make tool calls..It can Make

But in the case of langChain ,It will diretly make the tool calls.That is the difference .That's why we follow the function calling approach in langgarph

In [ ]:
response=llm_with_tools.invoke("what is the latest news on AI")
response.content